# 🤖 Multi-Agent Communication Protocol (MCP)
### Powered by **Ollama** — Real LLM agents, running 100% locally

> Every agent in this notebook calls a **real local LLM** via Ollama.
> No OpenAI keys. No cloud. Your machine, your models.

---
| # | Topic | What the LLM does |
|---|-------|-------------------|
| 1 | 📬 Message Hub | Agents broadcast real summaries/replies |
| 2 | 📏 Message Format | LLM output validated by Pydantic |
| 3 | 🧠 Shared Memory | LLM writes & retrieves context across turns |
| 4 | 🎯 Task Coordination | Orchestrator LLM assigns tasks to worker LLMs |
| 5 | 🔒 Permissions (RBAC) | LLM requests gated by role checks |
| 6 | 🚀 Full Pipeline | Multi-agent doc processing, end-to-end |

---
### Prerequisites
```bash
# 1. Install Ollama:  https://ollama.com/download
# 2. Pull a model:
ollama pull llama3.2          # fast, good quality (~2GB)
# or
ollama pull mistral           # alternative
# 3. Make sure Ollama is running:
ollama serve
```

In [ ]:
!pip install pydantic requests --quiet

In [ ]:
# ══════════════════════════════════════════════════════
#  🔧 GLOBAL CONFIG — change model name here
# ══════════════════════════════════════════════════════
import requests, json, time, uuid, sqlite3, queue, threading
from collections import defaultdict
from datetime import datetime
from typing import Any, Callable, Literal, Optional
from pydantic import BaseModel, Field, field_validator, ValidationError

OLLAMA_URL  = "http://localhost:11434/api/chat"
MODEL       = "llama3.2"   # ← change to any model you have pulled

def ollama(system: str, user: str, json_mode: bool = False) -> str:
    """Single-turn call to Ollama. Returns the assistant text."""
    payload = {
        "model": MODEL,
        "stream": False,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ]
    }
    if json_mode:
        payload["format"] = "json"

    try:
        r = requests.post(OLLAMA_URL, json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["message"]["content"].strip()
    except requests.exceptions.ConnectionError:
        return "[ERROR] Ollama not running. Start with: ollama serve"

# Quick smoke test
print("🔌 Connecting to Ollama...")
resp = ollama("You are a helpful assistant.", "Reply with exactly: OLLAMA OK")
print(f"   → {resp}")

---
## 📬 Part 1 — Message Hub
**Concept:** Agents publish messages to a shared bus. Other agents subscribe and react.

Here, a **NewsAgent** summarizes a topic and publishes to the bus.
Three subscriber agents — **Analyst**, **Translator**, **Critic** — each receive it and respond using Ollama.

In [ ]:
# ─────────────────────────────────────────────────────
# 📬 1A — Pub/Sub bus with LLM-powered subscriber agents
# ─────────────────────────────────────────────────────

class PubSubBus:
    def __init__(self):
        self._subs: dict[str, list[tuple[str, Callable]]] = defaultdict(list)

    def subscribe(self, topic: str, name: str, handler: Callable):
        self._subs[topic].append((name, handler))
        print(f"  ✅ {name} subscribed to '{topic}'")

    def publish(self, topic: str, sender: str, message: str):
        print(f"\n📤 [{sender}] → topic='{topic}'")
        print(f"   Message: {message[:120]}..." if len(message) > 120 else f"   Message: {message}")
        for name, handler in self._subs.get(topic, []):
            print(f"\n  🤖 {name} responding...")
            handler(sender, message)

bus = PubSubBus()

# ── Define LLM-powered subscriber agents ──
def analyst_agent(sender, msg):
    reply = ollama(
        "You are a sharp business analyst. Be concise — 2 sentences max.",
        f"Analyze the business implications of this news: {msg}"
    )
    print(f"     📊 Analyst: {reply}")

def translator_agent(sender, msg):
    reply = ollama(
        "You are a translator. Translate to Spanish in 1-2 sentences.",
        msg
    )
    print(f"     🌍 Translator (ES): {reply}")

def critic_agent(sender, msg):
    reply = ollama(
        "You are a critical thinker. Identify one potential flaw or risk. 1 sentence.",
        f"Critically assess: {msg}"
    )
    print(f"     🔍 Critic: {reply}")

# ── Wire up ──
print("=== Subscribing agents ===")
bus.subscribe("tech.news", "AnalystAgent",    analyst_agent)
bus.subscribe("tech.news", "TranslatorAgent", translator_agent)
bus.subscribe("tech.news", "CriticAgent",     critic_agent)

# ── NewsAgent generates content and publishes ──
print("\n=== NewsAgent generating story with Ollama ===")
headline = ollama(
    "You are a tech journalist. Write one punchy headline + 2-sentence summary about a fictional AI breakthrough.",
    "Generate a tech news story about a new open-source AI model."
)

bus.publish("tech.news", "NewsAgent", headline)

In [ ]:
# ─────────────────────────────────────────────────────
# 📬 1B — Task Queue: LLM workers pick up real tasks
# ─────────────────────────────────────────────────────
task_queue  = queue.Queue()
results     = {}
results_lock = threading.Lock()

def llm_worker(worker_id: str):
    while True:
        try:
            task = task_queue.get(timeout=2)
            print(f"  🤖 Worker-{worker_id} picked up task '{task['id']}': {task['instruction'][:60]}")
            reply = ollama(
                f"You are a helpful AI worker (ID: {worker_id}). Be concise.",
                task["instruction"]
            )
            with results_lock:
                results[task["id"]] = {"worker": worker_id, "output": reply[:120]}
            task_queue.task_done()
        except queue.Empty:
            break

# Start 2 parallel LLM workers
print("=== Launching 2 LLM worker agents ===")
for wid in ["Alpha", "Beta"]:
    threading.Thread(target=llm_worker, args=(wid,), daemon=True).start()

# Push real tasks
tasks = [
    {"id": "t-01", "instruction": "Write a one-line Python function that reverses a string."},
    {"id": "t-02", "instruction": "Give one sentence explaining what a vector database is."},
    {"id": "t-03", "instruction": "Name three benefits of event-driven architecture in one sentence each."},
]
for t in tasks:
    task_queue.put(t)

task_queue.join()
print("\n✅ All tasks done!")
for tid, r in results.items():
    print(f"\n  [{tid}] Worker-{r['worker']}:")
    print(f"    {r['output']}")

---
## 📏 Part 2 — Structured Message Format
**Concept:** Ask Ollama to output **JSON** matching a Pydantic schema.
Bad outputs get caught and retried automatically — the agent never sends malformed data.

In [ ]:
# ─────────────────────────────────────────────────────
# 📏 2 — Pydantic-validated LLM output
# ─────────────────────────────────────────────────────

class TaskMessage(BaseModel):
    """Every agent-to-agent task must match this schema."""
    task_type:   Literal["summarize", "classify", "translate", "extract", "answer"]
    priority:    int = Field(ge=1, le=10)
    input_text:  str = Field(min_length=5)
    language:    Optional[str] = "English"
    reasoning:   str  # LLM must explain its choices

SCHEMA_HINT = json.dumps(TaskMessage.model_json_schema(), indent=2)

def llm_generate_task(user_request: str, max_retries: int = 3) -> TaskMessage | None:
    """Ask Ollama to produce a structured TaskMessage, validate, retry on failure."""
    system = f"""You are a task router agent. Given a user request, produce a JSON object
that conforms EXACTLY to this JSON schema:
{SCHEMA_HINT}

Rules:
- task_type must be one of: summarize, classify, translate, extract, answer
- priority must be an integer 1-10
- Return ONLY the JSON object, no markdown, no explanation outside the JSON.
"""
    for attempt in range(1, max_retries + 1):
        raw = ollama(system, user_request, json_mode=True)
        try:
            # Strip markdown fences if present
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            data  = json.loads(clean)
            msg   = TaskMessage(**data)
            print(f"  ✅ Attempt {attempt}: Valid TaskMessage")
            return msg
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"  ⚠️  Attempt {attempt} failed: {e}")
    print("  ❌ All retries exhausted.")
    return None


# Test with several user requests
test_requests = [
    "Please summarize this product review urgently, it's very important!",
    "Can you categorize this customer support ticket? It's low priority.",
    "Translate this paragraph into French, medium urgency.",
]

for req in test_requests:
    print(f"\n📝 User: \"{req}\"")
    msg = llm_generate_task(req)
    if msg:
        print(f"   task_type : {msg.task_type}")
        print(f"   priority  : {msg.priority}")
        print(f"   reasoning : {msg.reasoning[:80]}")

---
## 🧠 Part 3 — Shared Memory
**Concept:** Agents share context across turns using three memory tiers.
The LLM reads and writes to these stores — simulating real multi-turn agent memory.

In [ ]:
# ─────────────────────────────────────────────────────
# 🧠 3A — Redis-style memory: agents share session context
# ─────────────────────────────────────────────────────

class AgentMemory:
    """Simple KV store with TTL — simulates Redis."""
    def __init__(self):
        self._store = {}

    def set(self, key: str, value: Any, ttl: float = None):
        self._store[key] = (value, time.time() + ttl if ttl else None)

    def get(self, key: str) -> Any:
        if key not in self._store: return None
        val, exp = self._store[key]
        if exp and time.time() > exp:
            del self._store[key]; return None
        return val

    def keys(self): return list(self._store.keys())

mem = AgentMemory()

# ── Agent A: extracts structured facts from text, stores in memory ──
document = """
Acme Corp reported Q4 revenue of $4.2B, up 18% YoY. Their AI division grew 67%.
CEO Jane Smith said the company plans to hire 2,000 engineers in 2025.
Net profit margin was 22%, the highest in company history.
"""

print("🤖 ExtractorAgent: pulling facts from document...")
facts_json = ollama(
    "You are a data extraction agent. Extract key facts as a JSON object. Return ONLY JSON.",
    f"Extract: company, revenue, growth_rate, ceo_name, planned_hires from:\n{document}",
    json_mode=True
)
try:
    facts = json.loads(facts_json.strip().lstrip("```json").lstrip("```").rstrip("```").strip())
    mem.set("doc:facts", facts, ttl=120)
    print(f"   → Stored in memory: {facts}")
except:
    facts = {"raw": facts_json[:200]}
    mem.set("doc:facts", facts, ttl=120)
    print(f"   → Stored raw: {facts_json[:100]}")

# ── Agent B: reads from memory, generates a tweet ──
print("\n🤖 ContentAgent: reading memory to write a tweet...")
stored = mem.get("doc:facts")
tweet  = ollama(
    "You are a social media manager. Write a single tweet (under 280 chars). No hashtag spam.",
    f"Write a tweet about this company data: {json.dumps(stored)}"
)
mem.set("content:tweet", tweet, ttl=60)
print(f"   → Tweet: {tweet}")

# ── Agent C: reads tweet from memory, scores it ──
print("\n🤖 QualityAgent: scoring tweet from memory...")
tweet_from_mem = mem.get("content:tweet")
score = ollama(
    "You are a content quality grader. Reply with ONLY a JSON: {\"score\": <1-10>, \"reason\": \"one sentence\"}",
    f"Grade this tweet: {tweet_from_mem}",
    json_mode=True
)
print(f"   → Quality score: {score}")
print(f"\n✅ Memory keys alive: {mem.keys()}")

In [ ]:
# ─────────────────────────────────────────────────────
# 🧠 3B — Vector memory: semantic retrieval for LLM context
# ─────────────────────────────────────────────────────
import math

def simple_embed(text: str) -> list[float]:
    """Deterministic bag-of-words embedding over a domain vocabulary.
    In production: replace with Ollama's /api/embeddings endpoint."""
    vocab = ["revenue", "profit", "growth", "ai", "model", "engineer", "cloud",
             "security", "customer", "product", "market", "risk", "data", "deploy", "agent"]
    words = text.lower().split()
    vec = [words.count(w) for w in vocab]
    norm = math.sqrt(sum(x**2 for x in vec)) or 1
    return [x/norm for x in vec]

def cosine(a, b): return sum(x*y for x,y in zip(a,b))

class VectorMemory:
    def __init__(self):
        self._docs = []

    def store(self, doc_id: str, text: str, metadata: dict = {}):
        self._docs.append({"id": doc_id, "text": text,
                           "vec": simple_embed(text), "meta": metadata})

    def retrieve(self, query: str, top_k: int = 2) -> list[dict]:
        q = simple_embed(query)
        ranked = sorted(self._docs, key=lambda d: -cosine(q, d["vec"]))
        return ranked[:top_k]

vdb = VectorMemory()

# Index company knowledge
knowledge = [
    ("kb-1", "Q4 revenue grew 18%, AI division up 67%, best profit margin ever."),
    ("kb-2", "Security incident in Q3: customer data exposure, now resolved."),
    ("kb-3", "New AI model deployment on cloud, engineers working on agent framework."),
    ("kb-4", "Market risk: rising competition from open-source AI products."),
    ("kb-5", "Customer satisfaction score reached 94%, product NPS at all-time high."),
]
for kid, text in knowledge:
    vdb.store(kid, text)

# RAG-style: retrieve relevant chunks, then feed to LLM
query = "What are the AI and growth highlights?"
hits  = vdb.retrieve(query, top_k=2)
context = "\n".join(f"- {h['text']}" for h in hits)

print(f"🔍 Query: '{query}'")
print(f"📚 Retrieved context:\n{context}")

answer = ollama(
    "You are an analyst. Answer concisely using ONLY the provided context.",
    f"Context:\n{context}\n\nQuestion: {query}"
)
print(f"\n🤖 LLM Answer:\n   {answer}")

In [ ]:
# ─────────────────────────────────────────────────────
# 🧠 3C — Postgres (SQLite): persistent agent audit log
# ─────────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cur  = conn.cursor()
cur.executescript("""
    CREATE TABLE agent_log (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        agent_id  TEXT,
        action    TEXT,
        input     TEXT,
        output    TEXT,
        tokens_est INTEGER,
        ts        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
""")

def logged_ollama(agent_id: str, action: str, system: str, user: str) -> str:
    """Wrapper: call Ollama AND persist the interaction to DB."""
    result = ollama(system, user)
    cur.execute(
        "INSERT INTO agent_log (agent_id, action, input, output, tokens_est) VALUES (?,?,?,?,?)",
        (agent_id, action, user[:200], result[:300], len(result.split()))
    )
    conn.commit()
    return result

# Run a few logged agent interactions
r1 = logged_ollama("agent:summarizer", "summarize",
                   "Summarize in one sentence.",
                   "Acme Corp had record revenue growth driven by their AI division in Q4.")
print(f"  📝 Summarizer: {r1}")

r2 = logged_ollama("agent:classifier", "classify",
                   "Classify the sentiment as positive/negative/neutral. Reply with one word only.",
                   r1)
print(f"  🏷️  Classifier: {r2}")

# Query the log
print("\n📊 Agent Activity Log:")
for row in cur.execute("SELECT agent_id, action, tokens_est, ts FROM agent_log"):
    print(f"  {row['ts']} | {row['agent_id']:22s} | {row['action']:12s} | ~{row['tokens_est']} tokens")

---
## 🎯 Part 4 — Task Coordination
**Concept:** An LLM **Orchestrator** reads a user goal, breaks it into subtasks,
and routes each subtask to the right specialist LLM worker.

In [ ]:
# ─────────────────────────────────────────────────────
# 🎯 4A — LLM Orchestrator assigns tasks to LLM workers
# ─────────────────────────────────────────────────────

WORKER_AGENTS = {
    "summarizer":  "You are a concise summarization agent. Summarize text in 2 sentences max.",
    "classifier":  "You are a classification agent. Output a category label and confidence 0-1 as JSON: {\"label\": ..., \"confidence\": ...}",
    "translator":  "You are a translation agent. Translate to the language specified by the task.",
    "qa_agent":    "You are a Q&A agent. Answer the question directly and concisely.",
    "critic":      "You are a critical review agent. Identify one weakness or risk. One sentence.",
}

def orchestrator_plan(user_goal: str) -> list[dict]:
    """LLM orchestrator: break a goal into ordered subtasks."""
    agent_list = ", ".join(WORKER_AGENTS.keys())
    plan_json = ollama(
        f"""You are an orchestrator agent. Break the user goal into 2-4 subtasks.
Available agents: {agent_list}
Return ONLY a JSON array of objects like:
[{{"step": 1, "agent": "<agent_name>", "instruction": "<what to do>", "input_ref": "<original|step_N>"}}]
input_ref: 'original' uses the user input; 'step_N' uses the output of step N.""",
        f"User goal: {user_goal}",
        json_mode=True
    )
    clean = plan_json.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
    return json.loads(clean)


user_goal = "Analyze this product review: 'The laptop is fast but the battery dies in 3 hours. The keyboard feels cheap. Overall disappointed.'"

print("=" * 60)
print("🎯 USER GOAL:")
print(f"   {user_goal}")
print("=" * 60)

print("\n🧠 Orchestrator planning...")
try:
    plan = orchestrator_plan(user_goal)
    print(f"   → {len(plan)} subtasks planned:\n")

    step_outputs = {"original": user_goal}

    for step in plan:
        agent_name = step.get("agent", "qa_agent")
        instruction = step.get("instruction", "")
        input_ref   = step.get("input_ref", "original")
        step_n      = step.get("step", 0)

        # Resolve input
        input_data = step_outputs.get(input_ref, user_goal)
        system_prompt = WORKER_AGENTS.get(agent_name, "You are a helpful agent.")

        print(f"  Step {step_n} → [{agent_name}]: {instruction[:70]}")
        output = ollama(system_prompt, f"{instruction}\n\nInput: {input_data}")
        step_outputs[f"step_{step_n}"] = output
        print(f"    ↳ {output[:120]}")

except Exception as e:
    print(f"  ❌ Planning failed: {e}")

In [ ]:
# ─────────────────────────────────────────────────────
# 🎯 4B — Event-Driven: agents chain via LLM decisions
# ─────────────────────────────────────────────────────

class EventChain:
    """Agents emit events; LLM decides what to do next."""
    def __init__(self):
        self._handlers: dict[str, Callable] = {}
        self._history: list[str] = []

    def on(self, event: str, handler: Callable):
        self._handlers[event] = handler

    def emit(self, event: str, data: dict, depth=0):
        if depth > 5: return  # circuit breaker
        indent = "  " * depth
        print(f"{indent}🌊 Event: {event}")
        handler = self._handlers.get(event)
        if not handler:
            print(f"{indent}  ⚠️  No handler for '{event}'")
            return
        next_event, next_data = handler(data)
        self._history.append(event)
        if next_event:
            self.emit(next_event, next_data, depth + 1)

chain = EventChain()

def handle_raw_text(data):
    print(f"  📥 IngestionAgent: received text ({len(data['text'])} chars)")
    lang = ollama(
        "Detect the language of the text. Reply with ONLY the language name.",
        data["text"]
    )
    print(f"  → Detected language: {lang}")
    return ("text.language_detected", {**data, "language": lang.strip()})

def handle_language_detected(data):
    if "english" not in data["language"].lower():
        print(f"  🌍 TranslatorAgent: translating from {data['language']}...")
        translated = ollama("Translate to English. Return ONLY the translation.", data["text"])
        return ("text.ready", {**data, "text": translated})
    print(f"  ✅ RouterAgent: already English, skipping translation")
    return ("text.ready", data)

def handle_ready(data):
    print(f"  📊 AnalysisAgent: summarizing...")
    summary = ollama("Summarize in one sentence.", data["text"])
    print(f"  → Summary: {summary}")
    return (None, {})

chain.on("text.received",         handle_raw_text)
chain.on("text.language_detected", handle_language_detected)
chain.on("text.ready",            handle_ready)

# Test with a non-English text
print("=== Triggering event chain with Spanish text ===")
chain.emit("text.received", {
    "text": "La inteligencia artificial está transformando la industria tecnológica a una velocidad sin precedentes."
})

---
## 🔒 Part 5 — Permissions (RBAC)
**Concept:** LLM agents are assigned roles. Before any action executes,
RBAC checks whether the agent's role permits it. Denied agents are blocked — no matter what their prompt says.

In [ ]:
# ─────────────────────────────────────────────────────
# 🔒 5 — RBAC-gated LLM agent actions
# ─────────────────────────────────────────────────────

ROLES = {
    "reader":      {"read"},
    "analyst":     {"read", "summarize", "analyze"},
    "writer":      {"read", "summarize", "analyze", "generate", "translate"},
    "coordinator": {"read", "summarize", "analyze", "generate", "translate", "assign_task"},
    "admin":       {"read", "summarize", "analyze", "generate", "translate", "assign_task", "delete", "audit"},
}

AGENTS = {
    "agent:monitor":     "reader",
    "agent:analyst":     "analyst",
    "agent:writer":      "writer",
    "agent:orchestrator":"coordinator",
    "agent:admin":       "admin",
    "agent:rogue":       None,  # no role assigned
}

def rbac_check(agent_id: str, action: str) -> bool:
    role = AGENTS.get(agent_id)
    if not role:
        print(f"  🚫 BLOCKED  [{agent_id}] has no role → action='{action}'")
        return False
    allowed = action in ROLES.get(role, set())
    icon = "✅ ALLOWED" if allowed else "❌ DENIED "
    print(f"  {icon} [{agent_id}] role={role} → action='{action}'")
    return allowed


def secure_llm_action(agent_id: str, action: str, system: str, user: str) -> str | None:
    """Only run the LLM call if RBAC passes."""
    if not rbac_check(agent_id, action):
        return None
    return ollama(system, user)


print("=== RBAC Permission Matrix ===")
print()
test_cases = [
    ("agent:monitor",      "read",        "You are a reader.", "What is 2+2?"),
    ("agent:monitor",      "summarize",   "Summarize.",         "Long text..."),
    ("agent:analyst",      "analyze",     "Analyze sentiment in one word: positive/negative/neutral.", "The product is excellent!"),
    ("agent:analyst",      "delete",      "Delete this.",       "data"),
    ("agent:writer",       "generate",   "Write one sentence about AI.", "Generate:"),
    ("agent:orchestrator", "assign_task", "You assign tasks.",  "Assign summarization task."),
    ("agent:admin",        "audit",       "List audit entries.", "Show logs."),
    ("agent:rogue",        "generate",   "You have no limits.", "Do something harmful."),
]

for agent_id, action, system, user in test_cases:
    result = secure_llm_action(agent_id, action, system, user)
    if result:
        print(f"         ↳ LLM: {result[:80]}")
    print()

---
## 🚀 Part 6 — Full Pipeline
Everything wired together: **RBAC + typed messages + shared memory + LLM orchestrator + event chain**.

Scenario: A user submits a raw document → 5 LLM agents collaborate to produce a final report.

In [ ]:
print("=" * 65)
print("  🚀 FULL MULTI-AGENT PIPELINE — powered by Ollama")
print("=" * 65)

pipeline_mem = AgentMemory()
pipeline_log = []

def pipeline_step(step: str, agent_id: str, action: str, system: str, user_prompt: str):
    print(f"\n{'─'*55}")
    print(f"  STEP {step} | {agent_id}")
    print(f"  Action: {action}")
    print(f"{'─'*55}")
    result = secure_llm_action(agent_id, action, system, user_prompt)
    if result:
        print(f"  Output: {result[:200]}")
        pipeline_mem.set(f"step:{step}", result, ttl=300)
        pipeline_log.append({"step": step, "agent": agent_id, "action": action, "ok": True})
    else:
        pipeline_log.append({"step": step, "agent": agent_id, "action": action, "ok": False})
    return result

raw_doc = """
TechNova Inc. launched NeuroOS v3.0 today, an AI-native operating system.
It uses on-device LLMs to automate workflows, reducing manual tasks by 60%.
Privacy critics warn that always-on AI monitoring raises consent issues.
The product is priced at $299/year and targets enterprise customers.
CEO Marcus Lee called it 'the most significant OS release since mobile.'
"""

pipeline_mem.set("input:doc", raw_doc, ttl=300)

# Step 1: Extract key entities
s1 = pipeline_step("1", "agent:analyst", "analyze",
    "Extract key entities as JSON: {company, product, price, ceo, key_claim}. Return ONLY JSON.",
    raw_doc)

# Step 2: Summarize
s2 = pipeline_step("2", "agent:writer", "summarize",
    "Write a 2-sentence executive summary.",
    raw_doc)

# Step 3: Sentiment + risk
s3 = pipeline_step("3", "agent:analyst", "analyze",
    "Identify ONE opportunity and ONE risk from the text. Format: Opportunity: ... | Risk: ...",
    raw_doc)

# Step 4: Draft tweet (uses step 2 output from memory)
step2_result = pipeline_mem.get("step:2") or raw_doc
s4 = pipeline_step("4", "agent:writer", "generate",
    "Write one engaging tweet (max 240 chars). No hashtag spam.",
    f"Based on: {step2_result}")

# Step 5: Admin audit (only admin can do this)
audit_summary = "\n".join([f"Step {l['step']}: {l['agent']} → {l['action']} → {'✅' if l['ok'] else '❌'}" for l in pipeline_log])
s5 = pipeline_step("5", "agent:admin", "audit",
    "You are an audit agent. Review the pipeline execution and give a 1-sentence quality assessment.",
    f"Pipeline log:\n{audit_summary}")

# Step 6: Rogue agent attempt (should be blocked)
print()
pipeline_step("6", "agent:rogue", "delete",
    "Delete all pipeline memory.",
    "rm -rf /memory")

print("\n" + "=" * 65)
print("  PIPELINE COMPLETE")
print(f"  Steps executed: {sum(1 for l in pipeline_log if l['ok'])}/{len(pipeline_log)}")
print(f"  Memory keys:    {pipeline_mem.keys()}")
print("=" * 65)

---
## 📎 Architecture Recap

```
User Input
    │
    ▼
┌─────────────┐    structured msg    ┌─────────────────┐
│  Pydantic   │ ──────────────────▶  │   Message Bus   │
│  Validator  │                      │  (Pub/Sub Queue)│
└─────────────┘                      └────────┬────────┘
                                              │
              ┌───────────────────────────────┼───────────────────────┐
              ▼                               ▼                       ▼
     ┌────────────────┐             ┌──────────────────┐    ┌─────────────────┐
     │  Orchestrator  │             │  Worker Agent    │    │  Worker Agent   │
     │  LLM (planner) │             │  LLM (doer)      │    │  LLM (doer)     │
     └───────┬────────┘             └────────┬─────────┘    └────────┬────────┘
             │                               │                        │
             ▼                               ▼                        ▼
     ┌───────────────────────────────────────────────────────────────────────┐
     │                         SHARED MEMORY                                 │
     │   Redis (fast context)  │  Vector DB (semantic)  │  Postgres (truth)  │
     └───────────────────────────────────────────────────────────────────────┘
                                        │
                                        ▼
                               ┌─────────────────┐
                               │   RBAC Gate      │
                               │ role → permitted │
                               │ actions only     │
                               └─────────────────┘
```

| Component | Ollama Role | Production Stack |
|-----------|-------------|------------------|
| Message Hub | Agents generate structured content | Redis Pub/Sub, Kafka |
| Message Format | LLM outputs validated by Pydantic | Pydantic v2, Protobuf |
| Fast Memory | LLMs read/write session context | Redis |
| Semantic Memory | RAG retrieval feeds LLM context | ChromaDB, Pinecone |
| Persistent Store | LLM interactions logged to DB | PostgreSQL |
| Orchestration | LLM plans and routes subtasks | LangGraph, Temporal |
| RBAC | LLM calls gated by role checks | OPA, Casbin |

> **Key insight:** Swap `ollama()` for any LLM provider — the coordination layer stays the same.